In [0]:
from decimal import Decimal
from pyspark.sql.types import *
schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_name", StringType()),
    StructField("product", StringType()),
    StructField("quantity", IntegerType()),
    StructField("unit_price", FloatType()),
    StructField("order_date", StringType()),
    StructField("status", StringType())
])

In [0]:
data = [
       (1001, 'Alice Johnson',  'Laptop',         1, 1299.99, '2024-01-15', 'Delivered'),
    (1002, 'Bob Smith',     'Wireless Mouse',  3,   29.99, '2024-01-18', 'Delivered'),
    (1003, 'Carol White',   'Monitor',         2,  399.99, '2024-02-02', 'Shipped'),
    (1004, 'David Brown',   'Keyboard',        1,   89.99, '2024-02-10', 'Processing'),
    (1005, 'Eva Martinez',  'Headphones',      2,  149.99, '2024-02-14', 'Delivered'),
    (1006, 'Frank Lee',     'Webcam',          1,   79.99, '2024-03-01', 'Cancelled'),
    (1007, 'Grace Kim',     'USB-C Hub',       4,   49.99, '2024-03-05', 'Shipped'),
    (1008, 'Henry Wilson',  'SSD Drive',       2,  129.99, '2024-03-12', 'Processing')
]

In [0]:
df = spark.createDataFrame(data, schema=schema)

In [0]:
display(df)

order_id,customer_name,product,quantity,unit_price,order_date,status
1001,Alice Johnson,Laptop,1,1299.99,2024-01-15,Delivered
1002,Bob Smith,Wireless Mouse,3,29.99,2024-01-18,Delivered
1003,Carol White,Monitor,2,399.99,2024-02-02,Shipped
1004,David Brown,Keyboard,1,89.99,2024-02-10,Processing
1005,Eva Martinez,Headphones,2,149.99,2024-02-14,Delivered
1006,Frank Lee,Webcam,1,79.99,2024-03-01,Cancelled
1007,Grace Kim,USB-C Hub,4,49.99,2024-03-05,Shipped
1008,Henry Wilson,SSD Drive,2,129.99,2024-03-12,Processing


In [0]:
display(df.printSchema())

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: float (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:
df.write.format('delta').mode('overwrite').save('/Volumes/useastws/default/deltavolume/ordersdata1/')

In [0]:
%sql
select * from delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

order_id,customer_name,product,quantity,unit_price,order_date,status
1001,Alice Johnson,Laptop,1,1299.99,2024-01-15,Delivered
1002,Bob Smith,Wireless Mouse,3,29.99,2024-01-18,Delivered
1003,Carol White,Monitor,2,399.99,2024-02-02,Shipped
1004,David Brown,Keyboard,1,89.99,2024-02-10,Processing
1005,Eva Martinez,Headphones,2,149.99,2024-02-14,Delivered
1006,Frank Lee,Webcam,1,79.99,2024-03-01,Cancelled
1007,Grace Kim,USB-C Hub,4,49.99,2024-03-05,Shipped
1008,Henry Wilson,SSD Drive,2,129.99,2024-03-12,Processing


In [0]:
%sql
describe history delta.`/Volumes/useastws/default/deltavolume/ordersdata1`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-08-26T17:48:04Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(2513891002461587),0826-063441-ydade36q,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 8, numOutputBytes -> 2825)",null,Databricks-Runtime/16.4.x-scala2.13


column rename wont work by default. need to set a proprty ('delta.columnMapping.mode' = 'name')

In [0]:
%sql
alter table delta.`/Volumes/useastws/default/deltavolume/ordersdata1`
rename column quantity to qty

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4545980698230925>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'alter table delta.`/Volumes/useastws/default/deltavolume/ordersdata1`\nrename column quantity to qty\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:130, in SqlMagic.sql(self, line, cell)
    126     

In [0]:
%sql
ALTER TABLE delta.`/Volumes/useastws/default/deltavolume/ordersdata1` 
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')

In [0]:
%sql
alter table delta.`/Volumes/useastws/default/deltavolume/ordersdata1`
rename column quantity to qty

In [0]:
%sql
describe history delta.`/Volumes/useastws/default/deltavolume/ordersdata1`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-08-26T17:51:35Z,147836707444603,anooptu@gmail.com,RENAME COLUMN,"Map(oldColumnPath -> quantity, newColumnPath -> qty)",null,List(2513891002461587),0826-063441-ydade36q,1,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
1,2026-08-26T17:51:06Z,147836707444603,anooptu@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.columnMapping.mode"":""name""})",null,List(2513891002461587),0826-063441-ydade36q,0,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
0,2026-08-26T17:48:04Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(2513891002461587),0826-063441-ydade36q,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 8, numOutputBytes -> 2825)",null,Databricks-Runtime/16.4.x-scala2.13


In [0]:
%sql
describe formatted  delta.`/Volumes/useastws/default/deltavolume/ordersdata1/`

col_name,data_type,comment
order_id,int,null
customer_name,string,null
product,string,null
qty,int,null
unit_price,float,null
order_date,string,null
status,string,null
,,
# Delta Statistics Columns,,
Column Names,"customer_name, unit_price, product, quantity, order_id, order_date, status",


In [0]:
%sql
alter table delta.`/Volumes/useastws/default/deltavolume/ordersdata1`
drop column status

In [0]:
%sql
describe history delta.`/Volumes/useastws/default/deltavolume/ordersdata1`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-08-26T17:55:03Z,147836707444603,anooptu@gmail.com,DROP COLUMNS,"Map(columns -> [""status""])",null,List(2513891002461587),0826-063441-ydade36q,2,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
2,2026-08-26T17:51:35Z,147836707444603,anooptu@gmail.com,RENAME COLUMN,"Map(oldColumnPath -> quantity, newColumnPath -> qty)",null,List(2513891002461587),0826-063441-ydade36q,1,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
1,2026-08-26T17:51:06Z,147836707444603,anooptu@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.columnMapping.mode"":""name""})",null,List(2513891002461587),0826-063441-ydade36q,0,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.13
0,2026-08-26T17:48:04Z,147836707444603,anooptu@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(2513891002461587),0826-063441-ydade36q,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 8, numOutputBytes -> 2825)",null,Databricks-Runtime/16.4.x-scala2.13
